# Use ENA to download the files

In [21]:
import pandas as pd
import requests

In [22]:
df = pd.read_excel("metadata.xlsx")
commands = []

In [ ]:
def get_fastq_links(run_accession):
    """Query ENA API for real fastq_ftp links for a given run."""
    url = (
        "https://www.ebi.ac.uk/ena/portal/api/filereport"
        f"?accession={run_accession}&result=read_run&fields=fastq_ftp&format=tsv"
    )
    r = requests.get(url)
    if r.status_code != 200:
        print(f"⚠️ ENA API error for {run_accession}")
        return []
    lines = r.text.strip().split("\n")
    if len(lines) > 1 and lines[1].strip():
        return lines[1].split(";")
    return []

for idx, row in df.iterrows():
    '''
    okay hang on i think may mali ako
    dapat isama rin dito yung ena_sample or ena_experiement
    tapos ilagay sa case 2 "ERS" din 
    AGH BRUH UULITIN KO NA NAMAN PLSSSSSSSSSS
    '''
    country = str(row["country"]).replace(" ", "_")
    isolate = str(row["isolate name"]).replace(" ", "_")
    ena_run = str(row["ena_run"]).strip()
    aux_link = str(row.get("auxillary.ftp.link")).strip()

    # Case 1: If there is an auxillary link — use it directly.
    if pd.notnull(aux_link) and aux_link != "" and aux_link != "nan":
        # Make sure it has ftp:// prefix
        link = aux_link if aux_link.startswith("ftp://") else f"ftp://{aux_link}"
        output_name = f"{country}_{isolate}.fastq.gz"
        cmd = f"wget {link} -O {output_name}"
        commands.append(cmd)

    # Case 2: If there is a valid ENA run — get real fastq_ftp from ENA API.
    elif pd.notnull(ena_run) and ena_run.startswith(("ERR", "SRR", "DRR", "SAM")):
        links = get_fastq_links(ena_run)
        if links:
            for link in links:
                link_file = link.split("/")[-1]
                output_name = f"{country}_{ena_run}_{link_file}"
                cmd = f"wget ftp://{link} -O {output_name}"
                commands.append(cmd)
        else:
            print(f"⚠️ No fastq_ftp found for {ena_run}")

    else:
        print(f"⚠️ No valid FTP info for row {idx}: {row.to_dict()}")

⚠️ No valid FTP info for row 23: {'isolate name': 'site.20.subj.SCH8324997.lab.YA00134589.iso.1', 'country': 'South Africa', 'ena_project': nan, 'ena_sample': 'ERS6421606', 'ena_experiment': nan, 'ena_run': nan, 'auxillary.ftp.link': nan}
⚠️ No valid FTP info for row 54: {'isolate name': 'site.28.subj.1185.lab.R15574.iso.1', 'country': 'South Africa', 'ena_project': nan, 'ena_sample': 'SAMEA9070131', 'ena_experiment': nan, 'ena_run': nan, 'auxillary.ftp.link': nan}
⚠️ No valid FTP info for row 64: {'isolate name': 'site.10.subj.YA00068781.lab.YA00068781.iso.1', 'country': 'South Africa', 'ena_project': 'PRJEB44577', 'ena_sample': 'ERS6402937', 'ena_experiment': 'ERX5543596', 'ena_run': nan, 'auxillary.ftp.link': nan}
⚠️ No valid FTP info for row 69: {'isolate name': 'site.28.subj.975.lab.R11283.iso.1', 'country': 'South Africa', 'ena_project': nan, 'ena_sample': 'SAMEA9070009', 'ena_experiment': nan, 'ena_run': nan, 'auxillary.ftp.link': nan}
⚠️ No valid FTP info for row 78: {'isolate 

In [24]:
with open("download_fastqs.sh", "w") as f:
    for cmd in commands:
        f.write(cmd + "\n")

print("✅ Done: download_fastqs.sh ready.")

✅ Done: download_fastqs.sh ready.
